# Jerárquico con Process.hierarchical

Clasificación: **Proceso jerárquico.** Un manager LLM coordina especialistas que pueden delegarse trabajo entre sí.

## Diferencia con el notebook 4 (orchestrator)

En el notebook 4 usamos `Process.hierarchical` con una sola task. El manager decidía a quién delegar esa task única.

Aquí vamos un paso más allá: el manager tiene múltiples tasks que supervisar, y los agentes tienen `allow_delegation=True` para que puedan redelegar entre ellos. El manager decide el orden, asigna, y puede pedir correcciones.

## Cómo funciona

```python
crew = Crew(
    agents=[director, logistica, experiencias, transporte],
    tasks=[task_logistica, task_experiencias, task_transporte, task_final],
    process=Process.hierarchical,
    manager_llm="gpt-4o-mini",
)
```

El manager LLM:
1. Lee todas las tasks pendientes
2. Decide qué agente ejecuta cada una
3. Puede re-asignar si un agente no da un resultado satisfactorio
4. Los agentes con `allow_delegation=True` pueden pedir ayuda a otros agentes del crew

In [1]:
!uv pip install -r requirements.txt --quiet

In [2]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

In [3]:
from crewai import Agent, Task, Crew, Process

inputs = {"destino": "Islandia", "dias": 5, "personas": 2, "presupuesto": 2200}

# Agentes especializados con delegación habilitada
logistica = Agent(
    role="Coordinador de Logística",
    goal="Resolver vuelos y alojamiento dentro del presupuesto",
    backstory="Gestionas la parte logística de viajes: vuelos, transfers y alojamiento.",
    allow_delegation=True,
)

experiencias = Agent(
    role="Coordinador de Experiencias",
    goal="Diseñar actividades y experiencias para el destino",
    backstory="Conoces atracciones, restaurantes y experiencias locales. Puedes pedir ayuda al de transporte para coordinar desplazamientos.",
    allow_delegation=True,
)

transporte = Agent(
    role="Especialista en Transporte",
    goal="Proponer opciones de transporte entre puntos del viaje",
    backstory="Conoces bus, tren, taxi, coche de alquiler y ferry. Calculas tiempos y costes.",
    allow_delegation=False,
)

director = Agent(
    role="Director de Agencia",
    goal="Ensamblar un itinerario final coherente con todas las partes dentro del presupuesto",
    backstory="Recibes los reportes de logística, experiencias y transporte y los integras en un plan dia a dia.",
    allow_delegation=False,
)

# Tasks que el manager asignará
task_logistica = Task(
    description="Busca vuelos y alojamiento para {personas} personas, {dias} dias en {destino}. Presupuesto total: {presupuesto} EUR.",
    expected_output="Opciones de vuelo y alojamiento con precios.",
    agent=logistica,
)

task_experiencias = Task(
    description="Propón actividades para {dias} dias en {destino} para {personas} personas. Presupuesto total: {presupuesto} EUR.",
    expected_output="Plan de actividades dia por dia con coste estimado.",
    agent=experiencias,
)

task_transporte = Task(
    description="Propón opciones de transporte para moverse en {destino} durante {dias} dias. Presupuesto total: {presupuesto} EUR.",
    expected_output="Opciones de transporte con precio y tiempo.",
    agent=transporte,
)

task_final = Task(
    description="Con los reportes anteriores, ensambla un itinerario dia a dia dentro de {presupuesto} EUR para {personas} personas en {destino}.",
    expected_output="Itinerario final con vuelos, alojamiento, actividades, transporte, coste por partida y total.",
    agent=director,
    context=[task_logistica, task_experiencias, task_transporte],
)

crew = Crew(
    agents=[logistica, experiencias, transporte, director],
    tasks=[task_logistica, task_experiencias, task_transporte, task_final],
    process=Process.hierarchical,
    manager_llm="gpt-4o-mini",
    verbose=True,
)

result = await crew.kickoff_async(inputs=inputs)
print(result.raw)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.14                                                                                       │
│  Latest version:  1.15.16                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 90dc8e24-8543-4dc9-b5dc-1512c98d51b1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Busca vuelos y alojamiento para 2 personas, 5 dias en Islandia. Presupuesto total: 2200 EUR.             │
│  ID: 2ebd6de6-fb13-4086-b62c-017c6346eec9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Busca vuelos y alojamiento para 2 personas, 5 dias en Islandia. Presupuesto total: 2200 EUR.             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'Busca vuelos y alojamiento para 2 personas durante 5 días en Islandia con un presupuesto       │
│  total de 2200 EUR. Debes encontrar las mejores opciones que se ajusten a este presupuesto, asegurándo...       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Itinerario Final para Viaje a Islandia (2 personas)                                                            │
│  Fechas: 10 al 15 de septiembre de 2024                                                                         │
│  Duración: 5 noches / 6 días                                                                                    │
│  Presupuesto total: 2200 EUR                                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **1. Vuelos**                                                                                                  │
│  - Aerolínea: Icelandair (vuelo directo Madrid-Reikiavik)                                                       │
│  - Ida: 10/09/2024 – Salida 10:00 AM desde Madrid, llegada a Reikiavik aprox. 13:00                             │
│  - Vuelta: 15/09/2024 – Salida 14:00 PM desde Reikiavik, llegada a Madrid aprox. 17:00                          │
│  - Precio total para 2 personas ida y vuelta: 720 EUR                                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **2. Alojamiento**                                                                                             │
│  - Hotel: Fosshotel Reykjavik (3 estrellas, céntrico)                                                           │
│  - Tipo: Habitación doble estándar con desayuno buffet incluido                                                 │
│  - Valoración Booking: 8.6/10                                                                                   │
│  - Duración: 5 noches (del 10 al 15 de septiembre)                                                              │
│  - Precio total para 2 personas: 1100 EUR                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **3. Transfers Aeropuerto-Hotel-Aeropuerto**                                                                   │
│  - Servicio: Shuttle compartido                                                                                 │
│  - Costo ida y vuelta por persona: 50 EUR                                                                       │
│  - Total para 2 personas: 100 EUR                                                                               │
│                                                                                                                 │
│  ---                                                   

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Con los reportes anteriores, ensambla un itinerario dia a dia dentro de 2200 EUR para 2 personas en      │
│  Islandia.                                                                                                      │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Itinerario Final para Viaje a Islandia (2 personas)  
Fechas: 10 al 15 de septiembre de 2024  
Duración: 5 noches / 6 días  
Presupuesto total: 2200 EUR  

---

**1. Vuelos**  
- Aerolínea: Icelandair (vuelo directo Madrid-Reikiavik)  
- Ida: 10/09/2024 – Salida 10:00 AM desde Madrid, llegada a Reikiavik aprox. 13:00  
- Vuelta: 15/09/2024 – Salida 14:00 PM desde Reikiavik, llegada a Madrid aprox. 17:00  
- Precio total para 2 personas ida y vuelta: 720 EUR  

---  

**2. Alojamiento**  
- Hotel: Fosshotel Reykjavik (3 estrellas, céntrico)  
- Tipo: Habitación doble estándar con desayuno buffet incluido  
- Valoración Booking: 8.6/10  
- Duración: 5 noches (del 10 al 15 de septiembre)  
- Precio total para 2 personas: 1100 EUR  

---  

**3. Transfers Aeropuerto-Hotel-Aeropuerto**  
- Servicio: Shuttle compartido  
- Costo ida y vuelta por persona: 50 EUR  
- Total para 2 personas: 100 EUR  

---  

**4. Transporte Local**  
- Para desplazamientos dentro de Reikiavik y tours, se usarán

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 90dc8e24-8543-4dc9-b5dc-1512c98d51b1                                                                       │
│  Final Output: Itinerario Final para Viaje a Islandia (2 personas)                                              │
│  Fechas: 10 al 15 de septiembre de 2024                                                                         │
│  Duración: 5 noches / 6 días                                                                                    │
│  Presupuesto total: 2200 EUR                                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **1. Vuelos**                                                                                                  │
│  - Aerolínea: Icelandair (vuelo directo Madrid-Reikiavik)                                                       │
│  - Ida: 10/09/2024 – Salida 10:00 AM desde Madrid, llegada a Reikiavik aprox. 13:00                             │
│  - Vuelta: 15/09/2024 – Salida 14:00 PM desde Reikiavik, llegada a Madrid aprox. 17:00                          │
│  - Precio total para 2 personas ida y vuelta: 720 EUR                                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **2. Alojamiento**                                                                                             │
│  - Hotel: Fosshotel Reykjavik (3 estrellas, céntrico)                                                           │
│  - Tipo: Habitación doble estándar con desayuno buffet incluido                                                 │
│  - Valoración Booking: 8.6/10                                                                                   │
│  - Duración: 5 noches (del 10 al 15 de septiembre)                                                              │
│  - Precio total para 2 personas: 1100 EUR                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **3. Transfers Aeropuerto-Hotel-Aeropuerto**                                                                   │
│  - Servicio: Shuttle compartido                                                                                 │
│  - Costo ida y vuelta por persona: 50 EUR                                                                       │
│  - Total para 2 personas: 100 EUR                                                                               │
│                                                                                                                 │
│  ---                                                  

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Qué define este patrón

El manager LLM supervisa múltiples tasks y las asigna a los agentes. Los agentes con `allow_delegation=True` (logística y experiencias) pueden pedir ayuda a otros agentes del crew. Transporte y director no delegan, solo ejecutan.

Diferencia con el notebook 4: allí había una sola task general y el manager decidía a quién consultar. Aquí hay 4 tasks explícitas que el manager orquesta, y los agentes pueden colaborar entre sí (experiencias puede pedirle a transporte que calcule tiempos para coordinar actividades).